# Measuring the Similarity of Texts using TF-IDF

This notebook is modeled on the *Programming Historian* lesson [Understanding and Using Common Similarity Measures for Text Analysis](https://programminghistorian.org/en/lessons/common-similarity-measures) by John Ladd. Please visit this webpage for more explanation.



## I. Setup

### Ia. Import necessary libraries

In [1]:
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.stem import WordNetLemmatizer   ###
from nltk.corpus import stopwords
from nltk import RegexpTokenizer  
tokenizer = RegexpTokenizer(r'\w+')
stop = sorted(stopwords.words('english'))

In [2]:
import pathlib
from pathlib import Path
import glob 
import pandas as pd, numpy as np
from scipy.spatial.distance import pdist, squareform

## Ib. Read in text files and create a dataframe

In [3]:
textdir = Path("../../../data/sotu2")
pathlist = sorted(textdir.glob('*.txt')) 

In [4]:
tokenizer = RegexpTokenizer(r'\w+')
#n=50

txtList=[]
pathlist = sorted(textdir.glob('*.txt'))      # .glob only stores the pathlist temporarily (for some reason), so you need to call it again!2
for path in pathlist:
    fn=path.stem                       #stem returns the filename minus the ".txt" (file extension). 
    year, pres=fn.split("_")            # fn = "1794_Washington" becomes year = "1794" and pres = "Washington"
    with open(path,'r') as f:  
        text1 = f.read()                #opens each file and reads it in as "sotu"
    tokens=tokenizer.tokenize(text1)    # tokenizes "sotu"
    numtoks = len(tokens)             # counts the number of tokens in "sotu"
    ltokens_ns = [tok.lower() for tok in tokens if tok not in stop]
    txtList.append([pres, year, numtoks, tokens, ltokens_ns, text1])   #add this info for "sotu" to a running list for all sotu addresses
       

In [5]:
colnames=['pres','year','numtoks','tokens', 'ltoks_ns', 'fulltext']
textdf=pd.DataFrame(txtList, columns=colnames)  #places our completed list of SOTU info in a dataframe
textdf.head(10)                                #prints out the first 10 rows of this dataframe (the default value for head() is 5 rows)

,pres,year,numtoks,tokens,ltoks_ns,fulltext
0,Washington,1790,0,[],[],
1,Washington,1791,2314,"[Fellow, Citizens, of, the, Senate, and, House...","[fellow, citizens, senate, house, representati...",Fellow-Citizens of the Senate and House of Rep...
2,Washington,1792,2104,"[Fellow, Citizens, of, the, Senate, and, House...","[fellow, citizens, senate, house, representati...",Fellow-Citizens of the Senate and House of Rep...
3,Washington,1793,1973,"[Fellow, Citizens, of, the, Senate, and, House...","[fellow, citizens, senate, house, representati...",Fellow-Citizens of the Senate and House of Rep...
4,Washington,1794,2918,"[Fellow, Citizens, of, the, Senate, and, House...","[fellow, citizens, senate, house, representati...",Fellow-Citizens of the Senate and House of Rep...
5,Washington,1795,1988,"[Fellow, Citizens, of, the, Senate, and, House...","[fellow, citizens, senate, house, representati...",Fellow-Citizens of the Senate and House of Rep...
6,Washington,1796,2878,"[Fellow, Citizens, of, the, Senate, and, House...","[fellow, citizens, senate, house, representati...",Fellow-Citizens of the Senate and House of Rep...
7,Adams,1797,2060,"[Gentlemen, of, the, Senate, and, Gentlemen, o...","[gentlemen, senate, gentlemen, house, represen...",Gentlemen of the Senate and Gentlemen of the H...
8,Adams,1798,2218,"[Gentlemen, of, the, Senate, and, Gentlemen, o...","[gentlemen, senate, gentlemen, house, represen...",Gentlemen of the Senate and Gentlemen of the H...
9,Adams,1799,1505,"[Gentlemen, of, the, Senate, and, Gentlemen, o...","[gentlemen, senate, gentlemen, house, represen...",Gentlemen of the Senate and Gentlemen of the H...


In [6]:
textdf.sort_values(by = "year", ascending = False).head(10)

,pres,year,numtoks,tokens,ltoks_ns,fulltext
227,Trump,2018,5204,"[Mr, Speaker, Mr, Vice, President, Members, of...","[mr, speaker, mr, vice, president, members, co...","Mr. Speaker, Mr. Vice President, Members of Co..."
226,Trump,2017,5095,"[Thank, you, very, much, Mr, Speaker, Mr, Vice...","[thank, much, mr, speaker, mr, vice, president...","Thank you very much. Mr. Speaker, Mr. Vice Pre..."
225,Obama,2016,5628,"[Mr, Speaker, Mr, Vice, President, Members, of...","[mr, speaker, mr, vice, president, members, co...","Mr. Speaker, Mr. Vice President, Members of Co..."
224,Obama,2015,6961,"[Mr, Speaker, Mr, Vice, President, Members, of...","[mr, speaker, mr, vice, president, members, co...","Mr. Speaker, Mr. Vice President, Members of Co..."
223,Obama,2014,7017,"[Mr, Speaker, Mr, Vice, President, Members, of...","[mr, speaker, mr, vice, president, members, co...","Mr. Speaker, Mr. Vice President, Members of Co..."
222,Obama,2013,6607,"[Mr, Speaker, Mr, Vice, President, Members, of...","[mr, speaker, mr, vice, president, members, co...","Mr. Speaker, Mr. Vice President, Members of Co..."
221,Obama,2012,7204,"[Mr, Speaker, Mr, Vice, President, members, of...","[mr, speaker, mr, vice, president, members, co...","Mr. Speaker, Mr. Vice President, members of Co..."
220,Obama,2011,7117,"[Mr, Speaker, Mr, Vice, President, members, of...","[mr, speaker, mr, vice, president, members, co...","Mr. Speaker, Mr. Vice President, members of Co..."
219,Obama,2010,7252,"[Madame, Speaker, Vice, President, Biden, Memb...","[madame, speaker, vice, president, biden, memb...","Madame Speaker, Vice President Biden, Members ..."
218,Obama,2009,6021,"[Madame, Speaker, Mr, Vice, President, Members...","[madame, speaker, mr, vice, president, members...","Madame Speaker, Mr. Vice President, Members of..."


## II. Create a TF-IDF matrix

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import WordNetLemmatizer   ###

# Interface lemma tokenizer from nltk with sklearn
class LemmaTokenizer:                                               ###
    ignore_tokens = [',', '.', ';', ':', '"', '``', "''", '`']      ###
    def __init__(self):                                             ###
        self.wnl = WordNetLemmatizer()                              ###
    def __call__(self, doc):                                        ###
        #return [self.wnl.lemmatize(t) for t in word_tokenize(doc) if t not in self.ignore_tokens]
        return [self.wnl.lemmatize(t) for t in tokenizer.tokenize(doc) if t not in self.ignore_tokens]    ###
    
lemma_tokenizer = LemmaTokenizer()                                 ###
eng_stops = set(stopwords.words('english'))                        ###
lemma_stop = lemma_tokenizer(' '.join(eng_stops))   
tfidf_vectorizer3 = TfidfVectorizer(input = "filename", stop_words = lemma_stop, tokenizer = lemma_tokenizer)
tfidf_matrix = tfidf_vectorizer3.fit_transform(pathlist)


c:\Users\F0040RP\Documents\DartLib_RDS\python_workshops\py-essentials_ws\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [8]:
#cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
#print(cosine_sim)

## III. Measuring similarity



In [9]:
tfidf_array = tfidf_matrix.toarray()

In [10]:
textnamelist = [path.stem for path in pathlist]
euclidean_distances = pd.DataFrame(squareform(pdist(tfidf_array)), index=textnamelist, columns=textnamelist)
print(euclidean_distances)

                 1790_Washington  1791_Washington  1792_Washington  \
1790_Washington              0.0         1.000000         1.000000   
1791_Washington              1.0         0.000000         1.096377   
1792_Washington              1.0         1.096377         0.000000   
1793_Washington              1.0         1.102880         1.159830   
1794_Washington              1.0         1.168055         1.204996   
...                          ...              ...              ...   
2014_Obama                   1.0         1.327836         1.338297   
2015_Obama                   1.0         1.333702         1.337974   
2016_Obama                   1.0         1.337306         1.342212   
2017_Trump                   1.0         1.321842         1.324781   
2018_Trump                   1.0         1.342505         1.349739   

                 1793_Washington  1794_Washington  1795_Washington  \
1790_Washington         1.000000         1.000000         1.000000   
1791_Washington    

In [11]:
tgt = "2002_Bush"      #try plugging in the names of different SOTU addresses, to view possible choices, enter the following in a new code cell: `textnamelist`
top5_euclidean = euclidean_distances.nsmallest(10, tgt)[tgt][1:]
print(top5_euclidean)

2004_Bush          0.967391
2005_Bush          0.995409
2006_Bush          0.995676
1790_Washington    1.000000
2003_Bush          1.031643
2008_Bush          1.049079
2007_Bush          1.055801
2014_Obama         1.068538
1998_Clinton       1.071469
Name: 2002_Bush, dtype: float64


In [12]:
cosine_distances = pd.DataFrame(squareform(pdist(tfidf_array, metric='cosine')), index=textnamelist, columns=textnamelist)

top5_cosine = cosine_distances.nsmallest(6, tgt)[tgt][1:]
print(top5_cosine)

2004_Bush    0.467923
2005_Bush    0.495420
2006_Bush    0.495685
2003_Bush    0.532144
2008_Bush    0.550284
Name: 2002_Bush, dtype: float64
